# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library. The notebook guides you through metadata loading, record set inspection, field selection, and exploratory data analysis using Croissant-compliant methods that reference record sets, fields, and columns by their `@id` values.

### Dataset Source
The dataset is defined by a Croissant schema at:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed (uncomment if needed)
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")


## 2. Data Overview
Review available record sets and fields, referencing all by their `@id` for robust access:


In [ ]:
# Retrieve record sets from the dataset schema
record_sets = [rs['@id'] for rs in getattr(metadata, 'record_sets', [])]
if not record_sets:
    # Try to retrieve from the low-level JSON metadata if high-level attribute is absent
    record_sets = [rs.get('@id') for rs in getattr(metadata, 'to_json', lambda: {}).call().get('recordSet', [])]

if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Record set @id's in the dataset:")
    for rs_id in record_sets:
        print("  ", rs_id)
    # For demo, pick the first one for further analysis

# Display record set details using mlcroissant metadata structure
if record_sets:
    # Show available fields for the first record set
    first_rs_id = record_sets[0]
    rs_json = getattr(metadata, 'to_json', lambda: {}).call().get('recordSet', [])
    rec = next((rs for rs in rs_json if rs.get('@id') == first_rs_id), {})
    fields = rec.get('field', [])
    print(f"\nFields (`@id`) in record set {first_rs_id}:")
    for fld in fields:
        if isinstance(fld, dict):
            print("   ", fld.get('@id'))
        else:
            print("   ", fld)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

- For this dataset, record set and field `@id`s are used to access and extract data.
- Fields/columns are referenced by their `@id`, ensuring schema alignment.


In [ ]:
# For demonstration, let's extract all available record sets
# (Replace with your specific record set `@id` if known)

# Find all record sets
if not record_sets:
    # Try to autodetect record set @id's from the metadata JSON
    metadata_json = metadata.to_json()
    record_sets = [rs.get('@id') for rs in metadata_json.get('recordSet', [])]

# Store all loaded dataframes by record set @id
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Display dataframe for the primary record set (first one)
if record_sets:
    demo_rs_id = record_sets[0]
    if demo_rs_id in dataframes:
        print(f"\nColumns in record set {demo_rs_id}:\n", dataframes[demo_rs_id].columns.tolist())
        print("\nPreview:")
        display(dataframes[demo_rs_id].head())
    else:
        print(f"No data for record set {demo_rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply basic data cleansing and transformations—using numeric and categorical field `@id`s—such as filtering, normalization, or grouping. (Replace the example field `@id` as appropriate for your data.)


In [ ]:
# Adjust these field @id's as appropriate for your dataset. Here we pick likely numeric/categorical column IDs.

df = None
if record_sets:
    demo_rs_id = record_sets[0]
    df = dataframes[demo_rs_id]
    print(f"Sample columns in {demo_rs_id}:", df.columns.tolist())
    # You may inspect visually and set the `numeric_field_id` and `group_field_id` below

    # Example: Try to auto-detect a numeric field and a group/categorical field by name
    numeric_field_candidates = [col for col in df.columns if 'abundance' in col.lower() or 'count' in col.lower() or 'mw' in col.lower() or df[col].dtype != object]
    group_field_candidates = [col for col in df.columns if 'group' in col.lower() or 'condition' in col.lower() or 'sample' in col.lower() or 'desc' in col.lower()]

    # Fallback if auto-detection fails
    numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]
    group_field = group_field_candidates[0] if group_field_candidates else None

    # Clean and convert column to numeric if possible
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    # Filter by a threshold (10 as an arbitrary value)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for the filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # If a group/category field exists, group and aggregate
    if group_field is not None:
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
        )
        print(f"\nGrouped by '{group_field}':")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships. Example uses `matplotlib`, but you may use any standard Python plotting library.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If a group/categorical field exists, plot boxplot/grouped summary
    if group_field is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

- We loaded and inspected the FAIR^2 dataset's schema and content using the Croissant `@id` for every access.
- We extracted one or more record sets, explored their fields, and ran exploratory statistics directly from Croissant-compliant access patterns.
- You can tailor this notebook further by adjusting the `@id` values, applying richer filtering, or creating advanced visualizations for your downstream analyses!
